# Text Module


In [ ]:
import random
import re

import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.sparse import csr_matrix, hstack
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, ndcg_score
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, GPT2LMHeadModel, GPT2TokenizerFast, get_linear_schedule_with_warmup

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
device = "cuda"

## Загрузка таблиц с подписями и метриками


In [ ]:
metrics_df = pd.read_csv("captions_metrics.csv")
dataset_df = pd.read_csv("dataset_df.csv")

print(metrics_df.shape)
print(dataset_df.shape)

## Разделение данных на обучающую, валидационную и тестовую части


In [ ]:
image_ids = metrics_df["image_idx"].unique()
rng = np.random.RandomState(seed)
rng.shuffle(image_ids)

n_images = len(image_ids)
n_train = int(n_images * 0.8)
n_val = int(n_images * 0.1)

train_ids = set(image_ids[:n_train])
val_ids = set(image_ids[n_train:n_train + n_val])
test_ids = set(image_ids[n_train + n_val:])

train_df = metrics_df[metrics_df["image_idx"].isin(train_ids)].reset_index(drop = True)
val_df = metrics_df[metrics_df["image_idx"].isin(val_ids)].reset_index(drop = True)
test_df = metrics_df[metrics_df["image_idx"].isin(test_ids)].reset_index(drop = True)

print(f"train images: {len(train_ids)}, val images: {len(val_ids)}, test images: {len(test_ids)}")
print(f"train rows: {len(train_df)}, val rows: {len(val_df)}, test rows: {len(test_df)}")

## Функции оценки выбора подписи


In [ ]:
def evaluate_selection(df, score_col, target_col = "cider", group_col = "image_idx", k = 3):
    top1_ciders = []
    is_best_flags = []
    ndcgs = []

    for _, group in df.groupby(group_col):
        group = group.sort_values("caption_number")
        true_relevance = group[target_col].values.reshape(1, -1)
        pred_scores = group[score_col].values.reshape(1, -1)

        selected_idx = np.argmax(group[score_col].values)
        selected_row = group.iloc[selected_idx]
        top1_ciders.append(selected_row[target_col])
        is_best_flags.append(bool(selected_row["is_best_cider"]))

        if true_relevance.shape[1] > 1 and true_relevance.std() > 1e-8:
            ndcgs.append(ndcg_score(true_relevance, pred_scores, k = min(k, true_relevance.shape[1])))

    return {
        "avg_cider_selected": float(np.mean(top1_ciders)),
        "top1_accuracy": float(np.mean(is_best_flags)),
        f"ndcg@{k}": float(np.mean(ndcgs)) if ndcgs else None,
    }


def print_results(results):
    comparison_df = pd.DataFrame(results).T
    comparison_df = comparison_df[["avg_cider_selected", "top1_accuracy", "ndcg@3"]]
    comparison_df = comparison_df.sort_values("avg_cider_selected", ascending = False)
    print(comparison_df)

## Базовые методы сравнения


In [ ]:
results = {}

baseline_df = test_df.copy()
baseline_df["baseline_score"] = (baseline_df["caption_number"] == 1).astype(float)
results["baseline_first_caption"] = evaluate_selection(baseline_df, "baseline_score")

random_df = test_df.copy()
random_df["random_score"] = np.random.RandomState(seed).rand(len(random_df))
results["random"] = evaluate_selection(random_df, "random_score")
results["oracle"] = evaluate_selection(test_df, "cider")

print_results(results)

## Обучение кросс-энкодер модели


In [ ]:
model_name = "roberta-base"
max_len = 32
batch_size = 64

tokenizer = AutoTokenizer.from_pretrained(model_name)


class CaptionScoreDataset(Dataset):
    def __init__(self, df, tokenizer, max_len = 32, target_col = "cider"):
        self.texts = df["caption_text"].astype(str).tolist()
        self.targets = df[target_col].astype(float).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.texts[idx], truncation = True, max_length = self.max_len, padding = "max_length", return_tensors = "pt")
        item = {key: value.squeeze(0) for key, value in encoding.items()}
        item["target"] = torch.tensor(self.targets[idx], dtype = torch.float)
        return item


train_dataset = CaptionScoreDataset(train_df, tokenizer, max_len)
val_dataset = CaptionScoreDataset(val_df, tokenizer, max_len)
test_dataset = CaptionScoreDataset(test_df, tokenizer, max_len)

train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

In [ ]:
class CrossEncoderScorer(nn.Module):
    def __init__(self, model_name, dropout = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.head = nn.Sequential(nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_size // 2, 1))

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids = input_ids, attention_mask = attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        score = self.head(cls_embedding).squeeze(-1)
        return score


model = CrossEncoderScorer(model_name).to(device)

In [ ]:
epochs = 8
lr = 2e-5

optimizer = torch.optim.AdamW(model.parameters(), lr = lr, weight_decay = 0.01)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps = int(0.1 * total_steps), num_training_steps = total_steps)
criterion = nn.MSELoss()


def run_epoch(loader, train = True):
    model.train() if train else model.eval()
    total_loss = 0.0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        target = batch["target"].to(device)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            pred = model(input_ids, attention_mask)
            loss = criterion(pred, target)

        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        total_loss += loss.item() * len(target)

    return total_loss / len(loader.dataset)


best_val_loss = float("inf")
best_state = None

for epoch in range(epochs):
    train_loss = run_epoch(train_loader, train = True)
    val_loss = run_epoch(val_loader, train = False)
    print(f"epoch {epoch + 1}/{epochs} | train_loss = {train_loss:.5f} | val_loss = {val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {key: value.cpu().clone() for key, value in model.state_dict().items()}

model.load_state_dict(best_state)
torch.save(model.state_dict(), "cross_encoder_scorer.pt")
print("best val_loss:", best_val_loss)

In [ ]:
@torch.no_grad()
def predict_scores(df, loader):
    model.eval()
    preds = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pred = model(input_ids, attention_mask)
        preds.extend(pred.cpu().numpy().tolist())

    out_df = df.copy()
    out_df["model_score"] = preds
    return out_df


test_df_with_preds = predict_scores(test_df, test_loader)
test_df_with_preds.to_csv("test_predictions_cross_encoder.csv", index = False)

results["cross_encoder"] = evaluate_selection(test_df_with_preds, "model_score")
print_results(results)


## Обучение би-энкодер модели


In [ ]:
encoder_name = "all-MiniLM-L6-v2"
sbert = SentenceTransformer(encoder_name, device = str(device))


def encode_captions(df, sbert, batch_size = 128):
    texts = df["caption_text"].astype(str).tolist()
    embeddings = sbert.encode(texts, batch_size = batch_size, show_progress_bar = True, convert_to_numpy = True, normalize_embeddings = True)
    return embeddings


train_embeddings = encode_captions(train_df, sbert)
val_embeddings = encode_captions(val_df, sbert)
test_embeddings = encode_captions(test_df, sbert)

print(train_embeddings.shape, val_embeddings.shape, test_embeddings.shape)
torch.cuda.empty_cache()

In [ ]:
class EmbeddingScoreDataset(Dataset):
    def __init__(self, embeddings, targets):
        self.embeddings = torch.tensor(embeddings, dtype = torch.float32)
        self.targets = torch.tensor(targets, dtype = torch.float32)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return {"embedding": self.embeddings[idx], "target": self.targets[idx]}


train_emb_dataset = EmbeddingScoreDataset(train_embeddings, train_df["cider"].values)
val_emb_dataset = EmbeddingScoreDataset(val_embeddings, val_df["cider"].values)
test_emb_dataset = EmbeddingScoreDataset(test_embeddings, test_df["cider"].values)

emb_batch_size = 128
train_emb_loader = DataLoader(train_emb_dataset, batch_size = emb_batch_size, shuffle = True)
val_emb_loader = DataLoader(val_emb_dataset, batch_size = emb_batch_size, shuffle = False)
test_emb_loader = DataLoader(test_emb_dataset, batch_size = emb_batch_size, shuffle = False)

In [ ]:
class BiEncoderHead(nn.Module):
    def __init__(self, input_dim = 384, hidden_dim = 128, dropout = 0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)


bi_model = BiEncoderHead(input_dim = train_embeddings.shape[1]).to(device)

In [ ]:
bi_epochs = 50
bi_lr = 1e-3

optimizer = torch.optim.AdamW(bi_model.parameters(), lr = bi_lr, weight_decay = 1e-4)
criterion = nn.MSELoss()


def run_bi_epoch(loader, train = True):
    bi_model.train() if train else bi_model.eval()
    total_loss = 0.0

    for batch in loader:
        emb = batch["embedding"].to(device)
        target = batch["target"].to(device)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            pred = bi_model(emb)
            loss = criterion(pred, target)

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * len(target)

    return total_loss / len(loader.dataset)


best_val_loss = float("inf")
best_state = None

for epoch in range(bi_epochs):
    train_loss = run_bi_epoch(train_emb_loader, train = True)
    val_loss = run_bi_epoch(val_emb_loader, train = False)

    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"epoch {epoch + 1}/{bi_epochs} | train_loss = {train_loss:.5f} | val_loss = {val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {key: value.cpu().clone() for key, value in bi_model.state_dict().items()}

bi_model.load_state_dict(best_state)
torch.save(bi_model.state_dict(), "bi_encoder_head.pt")
print("best val_loss:", best_val_loss)


In [ ]:
@torch.no_grad()
def predict_bi_scores(df, loader):
    bi_model.eval()
    preds = []

    for batch in loader:
        emb = batch["embedding"].to(device)
        pred = bi_model(emb)
        preds.extend(pred.cpu().numpy().tolist())

    out_df = df.copy()
    out_df["bi_encoder_score"] = preds
    return out_df


test_df_with_preds = predict_bi_scores(test_df_with_preds, test_emb_loader)
test_df_with_preds.to_csv("test_predictions_with_bi_encoder.csv", index = False)

results["bi_encoder"] = evaluate_selection(test_df_with_preds, "bi_encoder_score")
print_results(results)

## Расчёт дополнительных текстовых признаков


In [ ]:
lm_name = "distilgpt2"
lm_tokenizer = GPT2TokenizerFast.from_pretrained(lm_name)
lm_model = GPT2LMHeadModel.from_pretrained(lm_name).to(device)
lm_model.eval()

if lm_tokenizer.pad_token is None:
    lm_tokenizer.pad_token = lm_tokenizer.eos_token

In [ ]:
@torch.no_grad()
def compute_perplexities(texts, model, tokenizer, device, batch_size = 64, max_len = 32):
    perplexities = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        encoding = tokenizer(batch_texts, return_tensors = "pt", padding = True, truncation = True, max_length = max_len)
        input_ids = encoding["input_ids"].to(device)
        attention_mask = encoding["attention_mask"].to(device)

        outputs = model(input_ids = input_ids, attention_mask = attention_mask, labels = input_ids)
        logits = outputs.logits[:, :-1, :]
        targets = input_ids[:, 1:]
        mask = attention_mask[:, 1:].float()

        log_probs = torch.log_softmax(logits, dim = -1)
        token_log_probs = log_probs.gather(2, targets.unsqueeze(-1)).squeeze(-1)
        token_log_probs = token_log_probs * mask

        seq_log_prob = token_log_probs.sum(dim = 1)
        seq_len = mask.sum(dim = 1).clamp(min = 1)
        avg_log_prob = seq_log_prob / seq_len
        perplexities.extend(torch.exp(-avg_log_prob).cpu().numpy().tolist())

    return perplexities


train_df["fluency_ppl"] = compute_perplexities(train_df["caption_text"].astype(str).tolist(), lm_model, lm_tokenizer, device)
val_df["fluency_ppl"] = compute_perplexities(val_df["caption_text"].astype(str).tolist(), lm_model, lm_tokenizer, device)
test_df["fluency_ppl"] = compute_perplexities(test_df["caption_text"].astype(str).tolist(), lm_model, lm_tokenizer, device)

train_df["caption_len"] = train_df["caption_text"].astype(str).apply(lambda text: len(text.split()))
val_df["caption_len"] = val_df["caption_text"].astype(str).apply(lambda text: len(text.split()))
test_df["caption_len"] = test_df["caption_text"].astype(str).apply(lambda text: len(text.split()))

print(train_df["fluency_ppl"].describe())

In [ ]:
feature_cols = ["caption_len", "fluency_ppl"]
feature_mean = train_df[feature_cols].mean()
feature_std = train_df[feature_cols].std()


def normalize_features(df):
    return ((df[feature_cols] - feature_mean) / feature_std).values.astype(np.float32)


train_features = normalize_features(train_df)
val_features = normalize_features(val_df)
test_features = normalize_features(test_df)

train_combined = np.concatenate([train_embeddings, train_features], axis = 1)
val_combined = np.concatenate([val_embeddings, val_features], axis = 1)
test_combined = np.concatenate([test_embeddings, test_features], axis = 1)

print(train_combined.shape)

## Обучение bi-encoder модели с текстовыми признаками


In [ ]:
train_emb_feat_dataset = EmbeddingScoreDataset(train_combined, train_df["cider"].values)
val_emb_feat_dataset = EmbeddingScoreDataset(val_combined, val_df["cider"].values)
test_emb_feat_dataset = EmbeddingScoreDataset(test_combined, test_df["cider"].values)

train_emb_feat_loader = DataLoader(train_emb_feat_dataset, batch_size = 128, shuffle = True)
val_emb_feat_loader = DataLoader(val_emb_feat_dataset, batch_size = 128, shuffle = False)
test_emb_feat_loader = DataLoader(test_emb_feat_dataset, batch_size = 128, shuffle = False)

bi_model_feat = BiEncoderHead(input_dim = train_combined.shape[1]).to(device)

In [ ]:
bi_feat_epochs = 30
optimizer = torch.optim.AdamW(bi_model_feat.parameters(), lr = 1e-3, weight_decay = 1e-4)
criterion = nn.MSELoss()


def run_bi_feat_epoch(loader, train = True):
    bi_model_feat.train() if train else bi_model_feat.eval()
    total_loss = 0.0

    for batch in loader:
        emb = batch["embedding"].to(device)
        target = batch["target"].to(device)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            pred = bi_model_feat(emb)
            loss = criterion(pred, target)

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * len(target)

    return total_loss / len(loader.dataset)


best_val_loss = float("inf")
best_state = None

for epoch in range(bi_feat_epochs):
    train_loss = run_bi_feat_epoch(train_emb_feat_loader, train = True)
    val_loss = run_bi_feat_epoch(val_emb_feat_loader, train = False)

    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"epoch {epoch + 1}/{bi_feat_epochs} | train_loss = {train_loss:.5f} | val_loss = {val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {key: value.cpu().clone() for key, value in bi_model_feat.state_dict().items()}

bi_model_feat.load_state_dict(best_state)
print("best val_loss:", best_val_loss)

In [ ]:
@torch.no_grad()
def predict_bi_feat_scores(df, loader):
    bi_model_feat.eval()
    preds = []

    for batch in loader:
        emb = batch["embedding"].to(device)
        pred = bi_model_feat(emb)
        preds.extend(pred.cpu().numpy().tolist())

    out_df = df.copy()
    out_df["bi_encoder_feat_score"] = preds
    return out_df


test_feat_preds = predict_bi_feat_scores(test_df, test_emb_feat_loader)
test_df_with_preds = test_df_with_preds.merge(test_feat_preds[["image_idx", "caption_number", "bi_encoder_feat_score"]], on = ["image_idx", "caption_number"], how = "left")

results["bi_encoder_with_features"] = evaluate_selection(test_df_with_preds, "bi_encoder_feat_score")
print_results(results)

## Оценка согласованности подписей внутри изображения


In [ ]:
sbert = SentenceTransformer("all-MiniLM-L6-v2", device = str(device))


def compute_consensus_scores(df, sbert, group_col = "image_idx", text_col = "caption_text"):
    out_rows = []

    for image_idx, group in df.groupby(group_col):
        group = group.reset_index(drop = True)
        texts = group[text_col].astype(str).tolist()
        embeddings = sbert.encode(texts, convert_to_numpy = True, normalize_embeddings = True)
        sim_matrix = embeddings @ embeddings.T
        np.fill_diagonal(sim_matrix, np.nan)
        group["consensus_score"] = np.nanmean(sim_matrix, axis = 1)
        out_rows.append(group)

    return pd.concat(out_rows, ignore_index = True)


test_df_with_preds = compute_consensus_scores(test_df_with_preds, sbert)

del sbert
if torch.cuda.is_available():
    torch.cuda.empty_cache()

results["consensus_reranking"] = evaluate_selection(test_df_with_preds, "consensus_score")
print_results(results)

## LLM as judge


In [ ]:
llm_name = "Qwen/Qwen2.5-3B-Instruct"
llm_dtype = torch.bfloat16 if device.type == "cuda" else torch.float32

llm_tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm_model = AutoModelForCausalLM.from_pretrained(llm_name, torch_dtype = llm_dtype).to(device)
llm_model.eval()

In [ ]:
sample_images = 400
test_image_ids = test_df["image_idx"].unique()
sample_images = min(sample_images, len(test_image_ids))

rng = np.random.RandomState(seed)
sampled_image_ids = rng.choice(test_image_ids, size = sample_images, replace = False)
llm_eval_df = test_df_with_preds[test_df_with_preds["image_idx"].isin(sampled_image_ids)].copy()

print(f"Строк: {len(llm_eval_df)}, изображений: {llm_eval_df['image_idx'].nunique()}")

In [ ]:
def build_ranking_prompt(captions):
    captions_block = "\n".join(f"{i + 1}. {caption}" for i, caption in enumerate(captions))
    prompt = (
        "You are given 5 candidate captions generated by an AI model for the SAME photo. "
        "You do not see the photo itself. Rank the captions from the most natural, fluent, "
        "and plausible description of a real photo to the least plausible.\n\n"
        f"{captions_block}\n\n"
        "Respond with ONLY a comma-separated list of the 5 numbers from best to worst. "
        "Example format: 3,1,4,2,5\n"
        "Your answer:"
    )
    return prompt


def parse_ranking(response, n = 5):
    numbers = re.findall(r"\d+", response)
    numbers = [int(x) for x in numbers if 1 <= int(x) <= n]
    seen = set()
    ordered = []

    for number in numbers:
        if number not in seen:
            ordered.append(number)
            seen.add(number)

    if sorted(ordered) != list(range(1, n + 1)):
        return list(range(1, n + 1))

    return ordered

In [ ]:
@torch.no_grad()
def get_llm_ranking(captions):
    prompt = build_ranking_prompt(captions)
    messages = [{"role": "user", "content": prompt}]
    text = llm_tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = True)
    inputs = llm_tokenizer(text, return_tensors = "pt").to(device)
    output = llm_model.generate(**inputs, max_new_tokens = 20, do_sample = False, pad_token_id = llm_tokenizer.eos_token_id)
    generated = output[0][inputs["input_ids"].shape[1]:]
    response = llm_tokenizer.decode(generated, skip_special_tokens = True)
    return parse_ranking(response, n = len(captions))


llm_scores_rows = []

for image_idx, group in tqdm(llm_eval_df.groupby("image_idx"), total = llm_eval_df["image_idx"].nunique()):
    group = group.sort_values("caption_number").reset_index(drop = True)
    captions = group["caption_text"].astype(str).tolist()
    ranking = get_llm_ranking(captions)
    rank_to_score = {cap_num: len(captions) - pos for pos, cap_num in enumerate(ranking)}

    for i, row in group.iterrows():
        llm_scores_rows.append({"image_idx": image_idx, "caption_number": row["caption_number"], "llm_judge_score": rank_to_score[i + 1]})

llm_scores_df = pd.DataFrame(llm_scores_rows)

In [ ]:
llm_eval_df = llm_eval_df.merge(llm_scores_df, on = ["image_idx", "caption_number"], how = "left")
llm_eval_df.to_csv("llm_judge_predictions.csv", index = False)

## Итоговое сравнение методов


In [ ]:
score_columns = {
    "oracle": "cider",
    "cross_encoder": "model_score",
    "bi_encoder": "bi_encoder_score",
    "bi_encoder_with_features": "bi_encoder_feat_score",
    "consensus_reranking": "consensus_score",
    "llm_judge": "llm_judge_score",
}

llm_eval_df["baseline_score"] = (llm_eval_df["caption_number"] == 1).astype(float)
llm_eval_df["random_score"] = np.random.RandomState(seed).rand(len(llm_eval_df))
score_columns["baseline_first_caption"] = "baseline_score"
score_columns["random"] = "random_score"

subsample_results = {}
for name, col in score_columns.items():
    subsample_results[name] = evaluate_selection(llm_eval_df, col)

print_results(subsample_results)